<a href="https://colab.research.google.com/github/teamepic043/Interactive-Campus-Info-Chatbot-AI-Agent/blob/main/final_draft_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Campus Query Agent



In [ ]:
# Cell 1 - Install dependencies
!pip install langchain langchain-google-genai langchain-community langgraph beautifulsoup4 requests pypdf gradio duckduckgo-search -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
# Cell 2 - Imports
import io
import json
import logging
import os
import re
from datetime import datetime
from urllib.parse import urljoin, urlparse
from typing import List, Optional

import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader
from IPython.display import Markdown, display
from duckduckgo_search import DDGS

logging.getLogger('pypdf').setLevel(logging.ERROR)

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.prebuilt import create_react_agent
from pydantic import BaseModel, Field

print('✅ Imports successful')

✅ Imports successful


In [ ]:
# Cell 3 - API Key
from google.colab import userdata
os.environ['GOOGLE_API_KEY'] = userdata.get('Gemini1')
print('✅ API key loaded')

✅ API key loaded


In [ ]:
# Cell 4 - LLM
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash-lite',
    temperature=0
)
print('✅ LLM ready')

✅ LLM ready


In [ ]:
# Cell 5 - Raw Web Scraping Utilities

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/117.0.0.0 Safari/537.36'
}

def _raw_fetch_text(url: str, timeout: int = 12) -> str:
    try:
        r = requests.get(url, headers=HEADERS, timeout=timeout, allow_redirects=True)
        r.raise_for_status()
        return BeautifulSoup(r.text, 'html.parser').get_text(separator=' ', strip=True)
    except requests.exceptions.SSLError:
        # Retry without SSL verification for colleges with self-signed certs
        r = requests.get(url, headers=HEADERS, timeout=timeout, verify=False, allow_redirects=True)
        return BeautifulSoup(r.text, 'html.parser').get_text(separator=' ', strip=True)

def _raw_fetch_links(url: str, timeout: int = 12) -> list:
    try:
        r = requests.get(url, headers=HEADERS, timeout=timeout, allow_redirects=True)
        r.raise_for_status()
    except requests.exceptions.SSLError:
        r = requests.get(url, headers=HEADERS, timeout=timeout, verify=False, allow_redirects=True)
    soup = BeautifulSoup(r.content, 'html.parser')
    results = []
    for a in soup.find_all('a', href=True):
        href = a['href'].strip()
        if not href or href.startswith(('#', 'javascript:')):
            continue
        abs_url = urljoin(url, href)
        text = a.get_text(separator=' ', strip=True)
        results.append({'href': href, 'abs_url': abs_url, 'text': text})
    return results

def _same_domain(base_url: str, target_url: str) -> bool:
    return urlparse(base_url).netloc == urlparse(target_url).netloc

def _extract_pdf_text(pdf_url: str, max_pages: int = 10) -> str:
    r = requests.get(pdf_url, headers=HEADERS, timeout=15)
    r.raise_for_status()
    reader = PdfReader(io.BytesIO(r.content))
    return '\n'.join(p.extract_text() or '' for p in reader.pages[:max_pages]).strip()

print('✅ Scraping utilities ready')

✅ Scraping utilities ready


In [ ]:
# Cell 6 - Disk Cache

PAGE_CACHE_FILE = '/content/page_cache.json'
_page_cache: dict = {}

def _load_page_cache():
    global _page_cache
    if os.path.exists(PAGE_CACHE_FILE):
        with open(PAGE_CACHE_FILE) as f:
            _page_cache = json.load(f)

def _save_page_cache():
    with open(PAGE_CACHE_FILE, 'w') as f:
        json.dump(_page_cache, f, indent=2)

def cached_page(url: str) -> str:
    if url not in _page_cache:
        print(f'    [HTTP  ] {url}')
        _page_cache[url] = _raw_fetch_text(url)
        _save_page_cache()
    else:
        print(f'    [cache ] {url}')
    return _page_cache[url]

def _sitemap_path(domain: str) -> str:
    return f"/content/sitemap_{domain.replace('.', '_')}.json"

def _load_sitemap(domain: str) -> Optional[dict]:
    path = _sitemap_path(domain)
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

def _save_sitemap(domain: str, sitemap: dict):
    with open(_sitemap_path(domain), 'w') as f:
        json.dump(sitemap, f, indent=2)

COLLEGE_URL_CACHE_FILE = '/content/college_url_cache.json'
_college_url_cache: dict = {}

def _load_college_url_cache():
    global _college_url_cache
    if os.path.exists(COLLEGE_URL_CACHE_FILE):
        with open(COLLEGE_URL_CACHE_FILE) as f:
            _college_url_cache = json.load(f)

def _save_college_url_cache():
    with open(COLLEGE_URL_CACHE_FILE, 'w') as f:
        json.dump(_college_url_cache, f, indent=2)

def clear_cache():
    global _page_cache, _college_url_cache
    _page_cache = {}
    _college_url_cache = {}
    files_to_remove = [PAGE_CACHE_FILE, COLLEGE_URL_CACHE_FILE] + [
        os.path.join('/content', f)
        for f in os.listdir('/content')
        if f.startswith('sitemap_') and f.endswith('.json')
    ]
    for f in files_to_remove:
        if os.path.exists(f):
            os.remove(f)
    print('All caches cleared.')

_load_page_cache()
_load_college_url_cache()
print('✅ Cache loaded')

✅ Cache loaded


In [ ]:
# Cell 7 - Pydantic Schemas & Category Metadata

class SitemapEntry(BaseModel):
    topic: str = Field(
        description='Topic category e.g. transport, placements, hostel, academics, fees, admissions, facilities, research, about, events, academic_calendar'
    )
    label: str = Field(description='Human-readable label for this link e.g. Bus Routes Page')
    url:   str = Field(description='Full absolute URL')
    kind:  str = Field(description='Either page or pdf')

class CollegeSitemap(BaseModel):
    entries: List[SitemapEntry] = Field(description='All classified links from the college site')

class CollegeLocation(BaseModel):
    address:   str = Field(description='Full street address of the college')
    city:      str = Field(description='City where the college is located')
    state:     str = Field(description='State or province')
    country:   str = Field(description='Country')
    pincode:   str = Field(description='PIN code or ZIP code, empty string if not found')
    map_query: str = Field(description='A Google Maps search query string for this address, e.g. "ANITS College Visakhapatnam Andhra Pradesh"')

class CollegeContact(BaseModel):
    emails:   List[str] = Field(description='List of email addresses found on the college website')
    phones:   List[str] = Field(description='List of phone/fax numbers found on the college website')
    website:  str = Field(description='Official website URL of the college')
    social:   List[str] = Field(description='List of social media profile URLs (LinkedIn, Twitter, Facebook, YouTube, Instagram)')
    summary:  str = Field(description='One-sentence summary of who to contact for admissions enquiries')

SEARCH_CATEGORIES = [
    'hostel', 'transport', 'academic_calendar', 'fees', 'placements',
    'admissions', 'facilities', 'research', 'events', 'academics', 'about', 'contact',
]

CATEGORY_META = {
    'hostel':            {'label': '🏠 Hostel',             'query': 'What hostel facilities are available?'},
    'transport':         {'label': '🚌 Transport',           'query': 'What are the bus/transport routes and timings?'},
    'academic_calendar': {'label': '📅 Academic Calendar',   'query': 'What is the academic calendar and important dates?'},
    'fees':              {'label': '💰 Fees',                'query': 'What is the fee structure?'},
    'placements':        {'label': '💼 Placements',          'query': 'What are the placement statistics and top recruiters?'},
    'admissions':        {'label': '📝 Admissions',          'query': 'What are the admission requirements and process?'},
    'facilities':        {'label': '🏛️ Facilities',          'query': 'What campus facilities are available?'},
    'research':          {'label': '🔬 Research',            'query': 'What research programs and publications are available?'},
    'events':            {'label': '🎉 Events',              'query': 'What upcoming events and fests are there?'},
    'academics':         {'label': '📚 Academics',           'query': 'What academic programs and courses are offered?'},
    'about':             {'label': 'ℹ️ About',               'query': 'Tell me about this college.'},
    'contact':           {'label': '📞 Contact',             'query': 'What are the contact details?'},
}

print('✅ Schemas ready')

✅ Schemas ready


In [ ]:
# Cell 8 - College URL Resolver (FIXED: restored Google fallback + better domain scoring)

SKIP_DOMAINS = (
    'wikipedia', 'facebook', 'linkedin', 'justdial',
    'shiksha', 'collegedunia', 'careers360', 'instagram',
    'twitter', 'youtube', 'indiamart', 'quora', 'naukri',
    'glassdoor', 'ambitionbox', 'placementseason', 'getmyuni',
    'collegesearch', 'studyabroad', 'reddit', 'topper',
)

# Known college URL overrides — add more as needed
KNOWN_COLLEGE_URLS = {
    'anits': 'https://www.anits.edu.in/',
    'anits college': 'https://www.anits.edu.in/',
    'anil neerukonda institute of technology and sciences': 'https://www.anits.edu.in/',
}

def _score_url(url: str, name_slug: str) -> int:
    """Score a URL by how likely it is the official college website (higher = better)."""
    score = 0
    domain = urlparse(url).netloc.lower()
    d_slug = domain.replace('.', '').replace('-', '').replace('_', '')
    if name_slug in d_slug or d_slug in name_slug:
        score += 10
    if domain.endswith('.edu.in') or domain.endswith('.ac.in'):
        score += 5
    elif domain.endswith('.edu'):
        score += 3
    elif domain.endswith('.org') or domain.endswith('.in'):
        score += 1
    if any(skip in domain for skip in SKIP_DOMAINS):
        score -= 100
    return score

def resolve_college_url(college_name: str) -> str:
    """
    Robustly resolve the official website for a college.
    Checks: known overrides → cache → DuckDuckGo → Google → domain guess.
    """
    key = college_name.strip().lower()

    # 1. Check known overrides first
    if key in KNOWN_COLLEGE_URLS:
        url = KNOWN_COLLEGE_URLS[key]
        print(f'  [known  ] {college_name} -> {url}')
        _college_url_cache[key] = url
        _save_college_url_cache()
        return url

    # 2. Check disk cache
    if key in _college_url_cache:
        print(f'  [cache  ] {college_name} -> {_college_url_cache[key]}')
        return _college_url_cache[key]

    print(f'  [lookup ] Searching official site for: {college_name}')
    name_slug = key.replace(' ', '').replace('.', '').replace("'", '')

    # 3. Strategy 1: DuckDuckGo
    ddg_candidates = []
    search_queries = [
        f'{college_name} official college website',
        f'{college_name} edu.in site',
        f'{college_name} college official site India',
    ]
    for query in search_queries:
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(query, max_results=8))
            for r in results:
                url = r.get('href', '')
                if url:
                    ddg_candidates.append(url)
        except Exception as e:
            print(f'  [DDG    ] Query failed: {e}')

    # Score and pick best DDG result
    if ddg_candidates:
        best = max(ddg_candidates, key=lambda u: _score_url(u, name_slug))
        if _score_url(best, name_slug) > -100:
            parsed = urlparse(best)
            base_url = f'{parsed.scheme}://{parsed.netloc}/'
            print(f'  [DDG    ] Best match: {base_url}')
            _college_url_cache[key] = base_url
            _save_college_url_cache()
            return base_url

    # 4. Strategy 2: Google search via requests (Commit 10 fallback restored)
    try:
        google_query = f'{college_name} official college website'
        google_url = f'https://www.google.com/search?q={google_query.replace(" ", "+")}'
        r = requests.get(google_url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')
        for a in soup.find_all('a', href=True):
            href = a['href']
            if href.startswith('/url?q='):
                actual = href.split('/url?q=')[1].split('&')[0]
                if _score_url(actual, name_slug) > 0:
                    parsed = urlparse(actual)
                    base_url = f'{parsed.scheme}://{parsed.netloc}/'
                    print(f'  [Google ] Found: {base_url}')
                    _college_url_cache[key] = base_url
                    _save_college_url_cache()
                    return base_url
    except Exception as e:
        print(f'  [Google ] Search failed: {e}')

    # 5. Strategy 3: Direct domain guessing (Commit 10 fallback restored)
    slug = name_slug[:15]
    for tld in ['edu.in', 'ac.in', 'edu', 'org.in']:
        guess = f'https://www.{slug}.{tld}/'
        try:
            r = requests.get(guess, headers=HEADERS, timeout=6)
            if r.status_code < 400:
                print(f'  [guess  ] Domain works: {guess}')
                _college_url_cache[key] = guess
                _save_college_url_cache()
                return guess
        except Exception:
            pass

    print(f'  [ERROR  ] Could not resolve URL for: {college_name}')
    return ''

print('✅ URL resolver ready (with Google + domain-guess fallback)')


✅ URL resolver ready (with Google + domain-guess fallback)


In [ ]:
# Cell 9 - Sitemap Builder (auto-called, no manual button needed)

sitemap_parser = PydanticOutputParser(pydantic_object=CollegeSitemap)

sitemap_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        'You are building a topic map of a college website.\n'
        'Given a list of links (anchor text | URL), classify each informational link into a topic.\n'
        'Topics: transport, placements, hostel, academics, fees, admissions, facilities, research, about, events, academic_calendar, contact, other.\n'
        'IMPORTANT: classify academic calendar, exam schedules, and semester dates under academic_calendar.\n'
        'Skip: navigation links, login, search, social media, external sites, duplicate entries.\n'
        'For PDFs set kind=pdf, for pages set kind=page.\n'
        'Respond ONLY with valid JSON matching this schema:\n'
        '{format_instructions}'
    ),
    HumanMessagePromptTemplate.from_template(
        'College: {college_name}\n'
        'Base URL: {base_url}\n\n'
        'Links found across the site (anchor text | full URL):\n'
        '{links_text}'
    )
])

sitemap_chain = sitemap_prompt | llm | sitemap_parser


def build_sitemap(college_name: str, base_url: str = '') -> dict:
    """
    Crawl 2 levels deep, classify links, save to disk.
    Returns the sitemap dict on success, or empty dict on failure.
    Auto-called by create_query — no manual button needed.
    """
    if not base_url:
        base_url = resolve_college_url(college_name)
        if not base_url:
            print(f'  [ERROR] Could not resolve URL for {college_name}')
            return {}

    domain = urlparse(base_url).netloc
    print(f'\n  Building sitemap for {college_name} ({domain})...')

    try:
        homepage_links = _raw_fetch_links(base_url)
    except Exception as e:
        print(f'  [ERROR] Homepage fetch failed: {e}')
        return {}

    all_links = list(homepage_links)
    visited = {base_url}
    sub_pages = [
        lnk for lnk in homepage_links
        if _same_domain(base_url, lnk['abs_url'])
        and not lnk['abs_url'].lower().endswith('.pdf')
        and lnk['abs_url'] not in visited
    ][:25]

    for lnk in sub_pages:
        sub_url = lnk['abs_url']
        visited.add(sub_url)
        try:
            sub_links = _raw_fetch_links(sub_url)
            all_links.extend(sub_links)
        except Exception:
            pass

    seen_urls = set()
    unique_links = []
    for lnk in all_links:
        abs_url = lnk['abs_url']
        if abs_url in seen_urls or not _same_domain(base_url, abs_url):
            continue
        seen_urls.add(abs_url)
        label = lnk['text'] or abs_url.split('/')[-1] or abs_url
        unique_links.append({'label': label, 'url': abs_url})

    print(f'  {len(unique_links)} unique links collected. Classifying...')

    links_text = '\n'.join(
        f"{lnk['label']} | {lnk['url']}"
        for lnk in unique_links[:80]
    )
    try:
        result = sitemap_chain.invoke({
            'college_name': college_name,
            'base_url': base_url,
            'links_text': links_text,
            'format_instructions': sitemap_parser.get_format_instructions()
        })
    except Exception as e:
        print(f'  [ERROR] LLM classification failed: {e}')
        return {}

    topics: dict = {}
    for entry in result.entries:
        t = entry.topic.lower()
        if t not in topics:
            topics[t] = []
        topics[t].append({'label': entry.label, 'url': entry.url, 'kind': entry.kind})

    sitemap = {
        'college_name': college_name,
        'base_url': base_url,
        'built_at': datetime.now().isoformat(),
        'topics': topics
    }

    _save_sitemap(domain, sitemap)
    print(f'  Sitemap saved. Topics: {list(topics.keys())}')
    return sitemap

print('✅ Sitemap builder ready')

✅ Sitemap builder ready


In [ ]:
# Cell 10 - Tool Registry (9 tools including NEW compare_colleges)
# FIXED: search_topic no-match message no longer sounds terminal
# FIXED: lookup_sitemap now also triggers web search hint on no match

_active_sitemap: dict = {}


@tool
def lookup_sitemap(topic: str) -> str:
    """
    Look up the pre-built college sitemap to find pages and PDFs for a given topic.
    ALWAYS call this tool FIRST. Do not call fetch_page before calling this.
    Input: a single topic word such as transport, placements, hostel, fees, academics,
           academic_calendar, admissions, facilities, research, events.
    Returns: labelled URLs classified under that topic, or the full sitemap if no match.
    """
    if not _active_sitemap:
        return 'No sitemap loaded. Navigate manually: call get_page_links on the base URL.'
    topics = _active_sitemap.get('topics', {})
    topic_lower = topic.lower().replace(' ', '_')
    matched = []
    for t, entries in topics.items():
        if topic_lower in t or t in topic_lower:
            for e in entries:
                matched.append(f"[{e['kind'].upper()}] {e['label']} -> {e['url']}")
    if matched:
        return f"Found {len(matched)} entries for '{topic}':\n" + '\n'.join(matched)
    # FIXED: don't say "no info" — show full sitemap so agent can still navigate
    lines = [
        f"No direct match for '{topic}' in sitemap. Showing all available topics below.",
        "You should: (1) try fetch_page on a relevant URL below, OR (2) call search_web as fallback.",
        ""
    ]
    for t, entries in topics.items():
        lines.append(f'[{t.upper()}]')
        for e in entries[:5]:
            lines.append(f"  {e['kind'].upper()} | {e['label']} -> {e['url']}")
    return '\n'.join(lines)


@tool
def fetch_page(url: str) -> str:
    """
    Fetch the visible text content of a college webpage.
    Use this after lookup_sitemap returns a relevant page URL.
    Results are cached - repeated calls to the same URL are free.
    Returns up to 4000 characters of page text.
    """
    try:
        return cached_page(url)[:4000]
    except Exception as e:
        return f'Error fetching {url}: {e}'


@tool
def get_page_links(url: str) -> str:
    """
    Get all links on a page as Anchor Text -> URL pairs.
    Use this to discover deeper pages or find PDF links the sitemap may have missed.
    Only returns links within the same college domain.
    Returns up to 40 links.
    """
    try:
        links = _raw_fetch_links(url)
        lines = [
            f"{lnk['text'] or '(no text)'} -> {lnk['abs_url']}"
            for lnk in links
            if _same_domain(url, lnk['abs_url'])
        ]
        lines = list(dict.fromkeys(lines))[:40]
        return '\n'.join(lines) if lines else 'No same-domain links found on this page.'
    except Exception as e:
        return f'Error getting links from {url}: {e}'


@tool
def read_pdf(url: str) -> str:
    """
    Download and extract text from a PDF document.
    Use this when lookup_sitemap or get_page_links reveals a PDF that likely
    contains the answer such as bus schedules, placement reports, fee structures.
    Returns up to 3000 characters of extracted text.
    """
    try:
        text = _extract_pdf_text(url)
        return text[:3000] if text else 'PDF appears to be empty or image-only.'
    except Exception as e:
        return f'Error reading PDF {url}: {e}'


@tool
def search_web(query: str) -> str:
    """
    Search the web using DuckDuckGo.
    Use this when lookup_sitemap or fetch_page did not return useful information,
    OR when the topic is not found in the sitemap after 2 tool attempts.
    Always include the college name in the query for best results.
    Input: a concise search query, e.g. 'ANITS college placement statistics 2024'.
    Returns: up to 5 search results with title, URL, and a short snippet.
    """
    try:
        results = []
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=5):
                results.append(
                    f"Title   : {r.get('title', 'N/A')}\n"
                    f"URL     : {r.get('href', 'N/A')}\n"
                    f"Snippet : {r.get('body', 'N/A')}\n"
                )
        if not results:
            return 'No results found for that query.'
        return f"Web search results for '{query}':\n\n" + '\n'.join(results)
    except Exception as e:
        return f'Web search failed: {e}'


@tool
def search_topic(topic: str) -> str:
    """
    Return all sitemap entries for a specific topic category AND fetch the
    first matching page/PDF to give an immediate preview of available info.
    Supported topics: hostel, transport, academic_calendar, fees, placements,
    admissions, facilities, research, events, academics, about, contact.
    Returns: list of links + first-page preview text (up to 2000 chars).
    If no sitemap match is found, it DOES NOT mean there is no information —
    call lookup_sitemap or search_web next.
    """
    if not _active_sitemap:
        return 'No sitemap loaded. Navigate manually using get_page_links on the base URL.'

    topics_map = _active_sitemap.get('topics', {})
    topic_key  = topic.lower().replace(' ', '_')

    matched_entries = []
    for t, entries in topics_map.items():
        if topic_key in t or t in topic_key:
            matched_entries.extend(entries)

    if not matched_entries:
        # FIXED: no longer says "no information" — directs agent to next steps
        return (
            f"Topic '{topic}' was not directly found in the pre-built sitemap. "
            f"Available sitemap topics: {list(topics_map.keys())}. "
            f"NEXT STEP: Call lookup_sitemap('{topic}') to search more broadly, "
            f"or call search_web('{_active_sitemap.get('college_name', 'college')} {topic}') "
            f"to search the web. Do NOT give up yet."
        )

    lines = [f"=== {topic.upper()} — {len(matched_entries)} sitemap entries ==="]
    for e in matched_entries[:10]:
        lines.append(f"  [{e['kind'].upper()}] {e['label']} -> {e['url']}")

    first_page = next((e for e in matched_entries if e['kind'] == 'page'), None)
    first_pdf  = next((e for e in matched_entries if e['kind'] == 'pdf'),  None)
    preview    = ''

    if first_page:
        try:
            content = cached_page(first_page['url'])[:2000]
            preview = f"\n--- Preview: {first_page['label']} ---\n{content}"
        except Exception as e:
            preview = f'\n(Could not fetch preview: {e})'
    elif first_pdf:
        try:
            content = _extract_pdf_text(first_pdf['url'])[:2000]
            preview = f"\n--- PDF Preview: {first_pdf['label']} ---\n{content}"
        except Exception as e:
            preview = f'\n(Could not fetch PDF preview: {e})'

    return '\n'.join(lines) + preview


# ── NEW TOOL: compare_colleges ────────────────────────────────────────────────

@tool
def compare_colleges(second_college_name: str, topic: str) -> str:
    """
    Compare the currently active college with a second college on a specific topic.
    This tool fetches data about the second college from the web and returns
    a side-by-side comparison of both colleges on the given topic.

    Use this when the user asks: 'Compare ANITS with IIT Bombay on placements',
    or 'How does this college compare to VIT on fees?' etc.

    Args:
        second_college_name: Name of the college to compare with (e.g. 'IIT Bombay')
        topic: The topic to compare on (e.g. 'placements', 'fees', 'hostel', 'admissions')

    Returns: Structured comparison with data points for both colleges.
    """
    if not _active_sitemap:
        return 'No active college loaded. Please enter a college name first.'

    college1  = _active_sitemap.get('college_name', 'Current College')
    base_url1 = _active_sitemap.get('base_url', '')

    # Gather data for College 1 from sitemap
    topics_map = _active_sitemap.get('topics', {})
    topic_key  = topic.lower().replace(' ', '_')
    college1_data = ''

    matched_entries = []
    for t, entries in topics_map.items():
        if topic_key in t or t in topic_key:
            matched_entries.extend(entries)

    if matched_entries:
        first_page = next((e for e in matched_entries if e['kind'] == 'page'), None)
        if first_page:
            try:
                college1_data = cached_page(first_page['url'])[:1500]
            except Exception:
                pass

    if not college1_data:
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(f'{college1} {topic}', max_results=3))
            college1_data = ' | '.join(r.get('body', '') for r in results)[:1500]
        except Exception:
            college1_data = f'No {topic} data found for {college1}.'

    # Gather data for College 2 from web search
    college2_data = ''
    college2_url = resolve_college_url(second_college_name)
    if college2_url:
        domain2 = urlparse(college2_url).netloc
        sitemap2 = _load_sitemap(domain2)
        if sitemap2:
            topics2 = sitemap2.get('topics', {})
            matched2 = []
            for t, entries in topics2.items():
                if topic_key in t or t in topic_key:
                    matched2.extend(entries)
            if matched2:
                first_page2 = next((e for e in matched2 if e['kind'] == 'page'), None)
                if first_page2:
                    try:
                        college2_data = cached_page(first_page2['url'])[:1500]
                    except Exception:
                        pass

    if not college2_data:
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(
                    f'{second_college_name} college {topic} India',
                    max_results=4
                ))
            college2_data = ' | '.join(r.get('body', '') for r in results)[:1500]
        except Exception:
            college2_data = f'No {topic} data found for {second_college_name}.'

    output = [
        f"## 🆚 Comparison: {college1} vs {second_college_name}",
        f"### Topic: {topic.upper()}",
        "",
        f"### 🏫 {college1}",
        college1_data or f'No {topic} data found on website.',
        "",
        f"### 🏫 {second_college_name}",
        college2_data or f'No {topic} data found.',
        "",
        "---",
        "*Note: Data sourced from official websites and web search. Verify with official sources.*"
    ]

    return '\n'.join(output)


print('✅ Core tools registered (lookup_sitemap, fetch_page, get_page_links, read_pdf, search_web, search_topic, compare_colleges)')


✅ Core tools registered (lookup_sitemap, fetch_page, get_page_links, read_pdf, search_web, search_topic, compare_colleges)


In [ ]:
# Cell 11 - Location & Contact Tools

location_parser = PydanticOutputParser(pydantic_object=CollegeLocation)

location_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        'Extract the physical address of the college from the provided webpage text.\n'
        'Look for address, location, contact us, reach us sections.\n'
        'Respond ONLY with valid JSON matching this schema:\n'
        '{format_instructions}'
    ),
    HumanMessagePromptTemplate.from_template(
        'College: {college_name}\n'
        'Page text:\n'
        '{page_text}'
    )
])

location_chain = location_prompt | llm | location_parser


def fetch_college_location(college_name: str, base_url: str) -> dict:
    candidates = []
    if _active_sitemap:
        topics = _active_sitemap.get('topics', {})
        for topic in ('about', 'contact', 'facilities', 'other'):
            for entry in topics.get(topic, [])[:2]:
                if entry['kind'] == 'page':
                    candidates.append(entry['url'])
    if base_url:
        candidates.append(base_url)

    combined_text = ''
    for url in candidates[:3]:
        try:
            combined_text += cached_page(url)[:2000] + '\n'
        except Exception:
            pass

    if not combined_text.strip():
        return {'address': 'Not found', 'city': '', 'state': '',
                'country': '', 'pincode': '', 'map_query': college_name}

    try:
        result = location_chain.invoke({
            'college_name': college_name,
            'page_text': combined_text[:4000],
            'format_instructions': location_parser.get_format_instructions()
        })
        return result.dict()
    except Exception as e:
        return {'address': f'Could not extract: {e}', 'city': '', 'state': '',
                'country': '', 'pincode': '', 'map_query': college_name}


@tool
def get_college_location(dummy: str = '') -> str:
    """
    Fetch the physical location / address of the college currently being queried.
    Use this when the student asks where the college is located, its address, or
    how to reach it.
    Returns a formatted address string.
    """
    if not _active_sitemap:
        return 'No active college loaded. Please ensure build_sitemap() was called first.'
    college_name = _active_sitemap.get('college_name', 'Unknown')
    base_url     = _active_sitemap.get('base_url', '')
    loc = fetch_college_location(college_name, base_url)
    parts = [loc['address']]
    if loc['city']:    parts.append(loc['city'])
    if loc['state']:   parts.append(loc['state'])
    if loc['pincode']: parts.append(loc['pincode'])
    if loc['country']: parts.append(loc['country'])
    return ', '.join(p for p in parts if p)


contact_parser = PydanticOutputParser(pydantic_object=CollegeContact)

contact_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        'Extract all contact details for the college from the provided webpage text.\n'
        'Include email addresses, phone/fax numbers, social media links, and the official website.\n'
        'For the summary field, write one sentence saying who to contact for admissions enquiries.\n'
        'Respond ONLY with valid JSON matching this schema:\n'
        '{format_instructions}'
    ),
    HumanMessagePromptTemplate.from_template(
        'College: {college_name}\n'
        'Pre-extracted emails (regex): {raw_emails}\n'
        'Pre-extracted phones (regex): {raw_phones}\n'
        'Page text:\n'
        '{page_text}'
    )
])

contact_chain = contact_prompt | llm | contact_parser


def _regex_extract_contacts(text: str) -> dict:
    emails = list(dict.fromkeys(
        re.findall(r'[\w.+-]+@[\w-]+\.[\w.-]+', text)
    ))[:10]
    phones = list(dict.fromkeys(
        re.findall(r'[\+\(]?[0-9][0-9 .\-\(\)]{7,}[0-9]', text)
    ))[:10]
    return {'emails': emails, 'phones': phones}


@tool
def get_contact_info(dummy: str = '') -> str:
    """
    Extract contact details for the college currently being queried.
    Returns email addresses, phone/fax numbers, social media links, and
    a brief summary of who to contact for admissions.
    Use this when a student asks for email, phone, contact details, or how
    to get in touch with the college.
    """
    if not _active_sitemap:
        return 'No active college loaded. Please ensure build_sitemap() was called first.'

    college_name = _active_sitemap.get('college_name', 'Unknown')
    base_url     = _active_sitemap.get('base_url', '')

    candidates = []
    if _active_sitemap:
        topics = _active_sitemap.get('topics', {})
        for topic in ('contact', 'about', 'admissions', 'other'):
            for entry in topics.get(topic, [])[:2]:
                if entry['kind'] == 'page':
                    candidates.append(entry['url'])
    if base_url:
        candidates.append(base_url)

    combined_text = ''
    for url in candidates[:3]:
        try:
            combined_text += cached_page(url)[:2000] + '\n'
        except Exception:
            pass

    if not combined_text.strip():
        return 'Could not fetch any pages to extract contact info.'

    raw = _regex_extract_contacts(combined_text)

    try:
        result = contact_chain.invoke({
            'college_name': college_name,
            'raw_emails': ', '.join(raw['emails']) or 'none found',
            'raw_phones': ', '.join(raw['phones']) or 'none found',
            'page_text': combined_text[:4000],
            'format_instructions': contact_parser.get_format_instructions()
        })
        lines = [f'**Contact Info for {college_name}**\n']
        if result.emails:
            lines.append('**Email(s):** ' + ', '.join(result.emails))
        if result.phones:
            lines.append('**Phone(s):** ' + ', '.join(result.phones))
        if result.website:
            lines.append(f'**Website:** {result.website}')
        if result.social:
            lines.append('**Social:** ' + ', '.join(result.social))
        if result.summary:
            lines.append(f'\n_{result.summary}_')
        return '\n'.join(lines)
    except Exception as e:
        if raw['emails'] or raw['phones']:
            return (
                f"Emails : {', '.join(raw['emails']) or 'not found'}\n"
                f"Phones : {', '.join(raw['phones']) or 'not found'}"
            )
        return f'Could not extract contact info: {e}'


# Final tool registry — 9 tools
TOOLS = [
    lookup_sitemap, fetch_page, get_page_links, read_pdf,
    search_web, search_topic, get_college_location,
    get_contact_info, compare_colleges
]
print(f'✅ All tools registered ({len(TOOLS)} total): {[t.name for t in TOOLS]}')

✅ All tools registered (9 total): ['lookup_sitemap', 'fetch_page', 'get_page_links', 'read_pdf', 'search_web', 'search_topic', 'get_college_location', 'get_contact_info', 'compare_colleges']


In [ ]:
# Cell 12 - ReAct Agent
# FIXED: System prompt rewritten to enforce search_web fallback
#        and prevent premature "no information" answers

AGENT_SYSTEM_PROMPT = (
    'You are a Campus Query Agent that answers student questions about colleges and universities.\n'
    'You have 9 tools. ALWAYS exhaust the website tools before concluding there is no information.\n'
    '\n'
    'MANDATORY TOOL SEQUENCE — follow this order:\n'
    '1. lookup_sitemap(topic) — ALWAYS call this FIRST for every query.\n'
    '   Extract the core topic word from the query (e.g. placements, fees, hostel, transport).\n'
    '2. fetch_page(url) — fetch the most relevant URL returned by lookup_sitemap.\n'
    '3. get_page_links(url) — if you need to go deeper or find PDFs on that page.\n'
    '4. read_pdf(url) — if a PDF is found that likely has the answer.\n'
    '5. search_topic(topic) — use for structured category browsing (hostel, transport,\n'
    '   academic_calendar, fees, placements, admissions, facilities, research, events, academics).\n'
    '6. get_college_location() — ONLY when the query is about the college address or location.\n'
    '7. get_contact_info() — ONLY when the student asks for email, phone, or contact details.\n'
    '8. compare_colleges(second_college, topic) — ONLY when the student wants to COMPARE colleges.\n'
    '9. search_web(query) — USE THIS after 3 tool calls with no useful result.\n'
    '   Include the college name in your search query, e.g. "ANITS placements 2024".\n'
    '\n'
    'CRITICAL RULES:\n'
    '- NEVER say "I have no information" or "I cannot find" without first calling search_web.\n'
    '- If lookup_sitemap returns "no match", you MUST still call fetch_page on the base URL\n'
    '  OR call search_web — do not give up.\n'
    '- If search_topic returns "not found in sitemap", immediately call search_web next.\n'
    '- After ANY 3 tool calls with no useful result from the website, call search_web.\n'
    '- search_web is NOT optional — it is a required fallback step before concluding.\n'
    '- Always include the college name in search_web queries.\n'
    '- Prefer official website info over web search results when both are available.\n'
    '- Format your final answer in clean markdown. Cite source URLs where possible.\n'
    '- Only after search_web also returns nothing may you say the information is unavailable.\n'
)

agent_graph = create_react_agent(llm, TOOLS)
print('✅ ReAct agent ready with 9 tools (fixed system prompt)')


✅ ReAct agent ready with 9 tools (fixed system prompt)


/tmp/ipykernel_443/3725221677.py:36: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_graph = create_react_agent(llm, TOOLS)


In [ ]:
# Cell 13 - Pipeline (auto-builds sitemap if not found)

def create_query(college_name: str, query: str, url: str = '') -> str:
    global _active_sitemap

    # Step 1: Resolve URL
    if not url:
        url = resolve_college_url(college_name)
        if not url:
            return (
                f'❌ Could not find the official website for "{college_name}".\n'
                f'Try using the full college name, e.g. "Anil Neerukonda Institute of Technology" instead of "ANITS".'
            )

    print(f"\n{'='*60}")
    print(f'Query   : {query}')
    print(f'College : {college_name} ({url})')
    print('='*60)

    domain = urlparse(url).netloc

    # Step 2: Load or AUTO-BUILD sitemap
    sitemap = _load_sitemap(domain)
    if sitemap:
        _active_sitemap = sitemap
        print(f"  ✅ Sitemap loaded: topics = {list(sitemap['topics'].keys())}")
    else:
        print(f'  ⚙️  No sitemap found — auto-building now (this takes ~30s, done once per college)...')
        sitemap = build_sitemap(college_name, url)
        if sitemap:
            _active_sitemap = sitemap
        else:
            # Fallback: use empty sitemap so agent can still navigate manually
            _active_sitemap = {'college_name': college_name, 'base_url': url, 'topics': {}}
            print('  ⚠️  Sitemap build failed. Agent will navigate manually.')

    # Step 3: Run the agent
    user_message = (
        f'College: {college_name}\n'
        f'Website: {url}\n'
        f'Query: {query}'
    )

    print('\n[Agent running...]')
    try:
        result = agent_graph.invoke({
            'messages': [
                SystemMessage(content=AGENT_SYSTEM_PROMPT),
                HumanMessage(content=user_message)
            ]
        })
    except Exception as e:
        return f'Agent error: {e}'

    print('\n[Agent trace]')
    tool_call_count = 0
    for msg in result['messages'][2:]:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                tool_call_count += 1
                print(f'  [{tool_call_count}] CALL   {tc["name"]}({tc["args"]})')
        elif type(msg).__name__ == 'ToolMessage':
            preview = (msg.content or '')[:100].replace('\n', ' ')
            print(f'      RESULT {preview}...')

    print(f'\n  Total tool calls: {tool_call_count}')
    return result['messages'][-1].content

print('✅ Pipeline ready (auto-builds sitemap on first query)')

✅ Pipeline ready (auto-builds sitemap on first query)


In [ ]:
# Cell 14 - Quick Test (ANITS — previously broken)
answer = create_query('ANITS', 'What are the placement statistics?')
display(Markdown(answer))

In [ ]:
# Cell 15 - Test the NEW compare_colleges tool
answer = create_query('ANITS', 'Compare ANITS with VIT Vellore on placements')
display(Markdown(answer))

In [ ]:
# Cell 16 - Gradio UI

import gradio as gr
import urllib.parse


def gradio_ask(college_name: str, query: str) -> str:
    if not college_name.strip() or not query.strip():
        return 'Please enter both a college name and a query.'
    return create_query(college_name.strip(), query.strip())


def gradio_get_location(college_name: str):
    if not college_name.strip():
        return 'Please enter a college name.', ''
    global _active_sitemap
    url = resolve_college_url(college_name.strip())
    if url:
        domain = urlparse(url).netloc
        sitemap = _load_sitemap(domain)
        if not sitemap:
            sitemap = build_sitemap(college_name.strip(), url)
        _active_sitemap = sitemap if sitemap else {'college_name': college_name.strip(), 'base_url': url, 'topics': {}}
    else:
        url = ''
        _active_sitemap = {'college_name': college_name.strip(), 'base_url': '', 'topics': {}}

    loc = fetch_college_location(college_name.strip(), url)
    parts = [loc['address']]
    if loc['city']:    parts.append(loc['city'])
    if loc['state']:   parts.append(loc['state'])
    if loc['pincode']: parts.append(loc['pincode'])
    if loc['country']: parts.append(loc['country'])
    address_str = ', '.join(p for p in parts if p and p != 'Not found')

    map_query     = loc.get('map_query') or college_name.strip()
    encoded_query = urllib.parse.quote(map_query)
    map_html = (
        f'<iframe '
        f'width="100%" height="350" style="border:0;border-radius:8px;" '
        f'loading="lazy" allowfullscreen '
        f'src="https://maps.google.com/maps?q={encoded_query}&output=embed">'
        f'</iframe>'
    )
    return address_str or 'Address not found on website.', map_html


def gradio_get_contact(college_name: str) -> str:
    if not college_name.strip():
        return 'Please enter a college name.'
    global _active_sitemap
    url = resolve_college_url(college_name.strip())
    if url:
        domain = urlparse(url).netloc
        sitemap = _load_sitemap(domain)
        if not sitemap:
            sitemap = build_sitemap(college_name.strip(), url)
        _active_sitemap = sitemap if sitemap else {'college_name': college_name.strip(), 'base_url': url, 'topics': {}}
    else:
        _active_sitemap = {'college_name': college_name.strip(), 'base_url': '', 'topics': {}}
    return get_contact_info('')


def gradio_smart_search(college_name: str, category: str) -> str:
    if not college_name.strip():
        return 'Please enter a college name at the top first.'
    meta  = CATEGORY_META.get(category, {})
    query = meta.get('query', f'Tell me about {category} at this college.')
    return create_query(college_name.strip(), query)


def gradio_smart_search_custom(college_name: str, custom_query: str) -> str:
    if not college_name.strip() or not custom_query.strip():
        return 'Please enter both a college name and a search query.'
    return create_query(college_name.strip(), custom_query.strip())


def gradio_compare(college1: str, college2: str, topic: str) -> str:
    """New compare function for the Compare tab."""
    if not college1.strip() or not college2.strip() or not topic.strip():
        return 'Please enter both college names and a topic to compare.'
    query = f'Compare {college1.strip()} with {college2.strip()} on {topic.strip()}'
    return create_query(college1.strip(), query)


def _make_category_click(cat_key: str):
    def _fn(college_name, _placeholder=''):
        return gradio_smart_search(college_name, cat_key)
    return _fn


# ── Gradio UI ─────────────────────────────────────────────────────────────────

with gr.Blocks(title='Campus Query Agent', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🎓 Campus Query Agent ')
    gr.Markdown(
        '**Step 1:** Enter a college name below.  \n'
        '**Step 2:** Use any tab — the sitemap is built **automatically** on first use (no manual button needed).'
    )

    with gr.Row():
        college_name = gr.Textbox(
            label='College Name',
            placeholder='e.g. ANITS or IIT Bombay',
            scale=4
        )
        clear_btn = gr.Button('🗑️ Clear Cache', variant='secondary', scale=1)

    gr.Markdown('---')

    with gr.Tabs():

        # ── Tab 1: Smart Search ────────────────────────────────────────────────
        with gr.TabItem('🔍 Smart Search'):
            gr.Markdown(
                '### Browse by Category\n'
                'Click any button to instantly search that topic on the college website.'
            )

            search_output = gr.Markdown(value='_Search results will appear here..._')

            with gr.Row():
                btn_hostel    = gr.Button('🏠 Hostel')
                btn_transport = gr.Button('🚌 Transport')
                btn_calendar  = gr.Button('📅 Academic Calendar')
                btn_fees      = gr.Button('💰 Fees')

            with gr.Row():
                btn_placements  = gr.Button('💼 Placements')
                btn_admissions  = gr.Button('📝 Admissions')
                btn_facilities  = gr.Button('🏛️ Facilities')
                btn_research    = gr.Button('🔬 Research')

            with gr.Row():
                btn_events    = gr.Button('🎉 Events')
                btn_academics = gr.Button('📚 Academics')
                btn_about     = gr.Button('ℹ️ About')
                btn_contact2  = gr.Button('📞 Contact')

            gr.Markdown('#### Or type your own search:')
            with gr.Row():
                custom_search_box = gr.Textbox(
                    label='',
                    placeholder='e.g. scholarships, library timings, sports facilities...',
                    scale=4
                )
                custom_search_btn = gr.Button('Search', variant='primary', scale=1)

            for btn, cat in [
                (btn_hostel,    'hostel'),
                (btn_transport, 'transport'),
                (btn_calendar,  'academic_calendar'),
                (btn_fees,      'fees'),
                (btn_placements,'placements'),
                (btn_admissions,'admissions'),
                (btn_facilities,'facilities'),
                (btn_research,  'research'),
                (btn_events,    'events'),
                (btn_academics, 'academics'),
                (btn_about,     'about'),
                (btn_contact2,  'contact'),
            ]:
                btn.click(
                    fn=_make_category_click(cat),
                    inputs=[college_name],
                    outputs=search_output
                )

            custom_search_btn.click(
                fn=gradio_smart_search_custom,
                inputs=[college_name, custom_search_box],
                outputs=search_output
            )

        # ── Tab 2: Ask a Query ─────────────────────────────────────────────────
        with gr.TabItem('💬 Ask a Query'):
            user_query = gr.Textbox(
                label='Your Query',
                placeholder='e.g. What are the bus routes?'
            )
            ask_btn = gr.Button('Ask Agent', variant='primary')
            output  = gr.Markdown(value='_Your answer will appear here..._')

            gr.Examples(
                examples=[
                    ['ANITS',      'What are the transportation routes?'],
                    ['ANITS',      'What are the placement statistics?'],
                    ['ANITS',      'What hostel facilities are available?'],
                    ['ANITS',      'What is the academic calendar?'],
                    ['IIT Bombay', 'What are the admission requirements?'],
                    ['ANITS',      'What is the email address for admissions?'],
                ],
                inputs=[college_name, user_query]
            )

            ask_btn.click(fn=gradio_ask, inputs=[college_name, user_query], outputs=output)

        # ── Tab 3: 🆕 Compare Colleges ─────────────────────────────────────────
        with gr.TabItem('🆚 Compare Colleges'):
            gr.Markdown(
                '### Compare Two Colleges\n'
                'Compare the college above with another college on any topic — '
                'placements, fees, hostel, admissions, facilities, etc.'
            )
            with gr.Row():
                compare_college2 = gr.Textbox(
                    label='Compare With (second college)',
                    placeholder='e.g. VIT Vellore or NIT Warangal'
                )
                compare_topic = gr.Dropdown(
                    label='Topic',
                    choices=[
                        'placements', 'fees', 'hostel', 'admissions',
                        'academics', 'facilities', 'transport', 'research',
                        'events', 'about'
                    ],
                    value='placements'
                )
            compare_btn = gr.Button('⚡ Compare Now', variant='primary')
            compare_output = gr.Markdown(value='_Comparison will appear here..._')

            gr.Examples(
                examples=[
                    ['ANITS', 'VIT Vellore',    'placements'],
                    ['ANITS', 'NIT Warangal',   'fees'],
                    ['ANITS', 'IIT Bombay',     'admissions'],
                    ['ANITS', 'JNTU Kakinada',  'hostel'],
                ],
                inputs=[college_name, compare_college2, compare_topic]
            )

            compare_btn.click(
                fn=gradio_compare,
                inputs=[college_name, compare_college2, compare_topic],
                outputs=compare_output
            )

        # ── Tab 4: College Location ────────────────────────────────────────────
        with gr.TabItem('📍 College Location'):
            gr.Markdown(
                '### Find the College Location\n'
                'Extracts the physical address from the college website and shows it on a map.'
            )
            location_btn = gr.Button('Get Location', variant='primary')
            address_out  = gr.Textbox(
                label='Extracted Address',
                interactive=False,
                placeholder='Address will appear here...'
            )
            map_out = gr.HTML(label='Map')

            location_btn.click(
                fn=gradio_get_location,
                inputs=[college_name],
                outputs=[address_out, map_out]
            )

        # ── Tab 5: Contact Info ────────────────────────────────────────────────
        with gr.TabItem('📞 Contact Info'):
            gr.Markdown(
                '### Get College Contact Details\n'
                'Extracts email addresses, phone numbers, and social media links from the college website.'
            )
            contact_btn = gr.Button('Get Contact Info', variant='primary')
            contact_out = gr.Markdown(value='_Contact details will appear here..._')

            contact_btn.click(
                fn=gradio_get_contact,
                inputs=[college_name],
                outputs=contact_out
            )

    clear_btn.click(fn=lambda: clear_cache())

if __name__ == '__main__':
    demo.launch()


/tmp/ipykernel_443/333745249.py:98: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='Campus Query Agent', theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f8ea28779a00d882d4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
